# RF-DETR pose · dataset-fixer
Use a Colab GPU runtime. Place the accompanying `dataset_fixer-0.1.0.dev0+training20260925-py3-none-any.whl` in `MyDrive/wolfs/rf-detr` before running the installation cell. This local framework build contains the unified training API.

Run the cells in order. Checkpoints upload to W&B and a verified bundle is copied to Drive after each epoch. Evaluation uses the best weights. Colab disconnects after all configured destinations confirm completion; failed uploads keep the runtime connected.

For Roboflow, set `ROBOFLOW_API_KEY` in Colab Secrets and grant notebook access. W&B login remains the usual command below.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%pip install -q "/content/drive/MyDrive/wolfs/rf-detr/dataset_fixer-0.1.0.dev0+training20260925-py3-none-any.whl[rfdetr,roboflow]"

In [ ]:
!wandb login

In [ ]:
import dataset_fixer as df

## Run settings
`DATASET_SOURCE` accepts a Drive folder/ZIP or a pinned `roboflow:workspace/project/version` source. Set `BASE_WEIGHTS` to a local checkpoint, bundle ZIP, or `wandb:entity/project/run-id` to start a new run from its best weights.

In [ ]:
DATASET_SOURCE = "roboflow:wolfcorrected/new_wolf/5"
WANDB_PROJECT = "wolf-poses"
BASE_WEIGHTS = None
RESOLUTION = 1296
EPOCHS = 50
BATCH_SIZE = 14
BACKUP_DIR = "/content/drive/MyDrive/wolfs/rf-detr"

## Augmentations
This configuration is shared by the native training pipeline and its preview. Keypoint shape and flip order come from the dataset.

In [ ]:
AUG_CONFIG = {
    "Affine": {"scale": (0.8, 2.0), "translate_percent": (-0.4, 0.4), "rotate": (-180, 180), "p": 0.7},
    "HueSaturationValue": {"hue_shift_limit": 18, "sat_shift_limit": 20, "val_shift_limit": 50, "p": 0.4},
    "RandomBrightnessContrast": {"brightness_limit": 0.15, "contrast_limit": 0.15, "p": 0.3},
    "GaussianBlur": {"blur_limit": (3, 5), "p": 0.1},
}

In [ ]:
dataset = df.Dataset.open(DATASET_SOURCE, deep=True)
TRAINING = df.TrainingConfig(
    resolution=RESOLUTION, epochs=EPOCHS, batch_size=BATCH_SIZE,
    workers=4, seed=7,
    backend_options={"lr": 1e-4, "lr_encoder": 1e-4, "grad_accum_steps": 1, "compute_val_loss": True},
)
dataset

In [ ]:
df.preview_augmentations(
    dataset, AUG_CONFIG, type=df.ModelTypes.RFDETR,
    weights=BASE_WEIGHTS, config=TRAINING, samples=3,
)

## Train, evaluate, upload, disconnect
A new W&B run is created each time. Best weights and resumable latest checkpoints are distinct. The session also finalizes after Python errors or interruption, then disconnects only if checkpoints are safe. Abrupt runtime termination can recover only checkpoints already published.

In [ ]:
with df.TrainingSession(
    wandb=df.WandbConfig(project=WANDB_PROJECT),
    checkpointing=df.CheckpointConfig(backup_dir=BACKUP_DIR),
    disconnect="after_safe", disconnect_delay=30,
) as session:
    result = df.train(
        dataset, type=df.ModelTypes.RFDETR, weights=BASE_WEIGHTS,
        config=TRAINING, augmentations=AUG_CONFIG, session=session,
    )
    evaluation = result.evaluate(samples=32, plots=6)
    display(evaluation.ranking)

If an upload failed, fix the connection and run `session.finish()` to retry without retraining.

To fine-tune a finished run at another resolution, set `BASE_WEIGHTS = "wandb:entity/project/run-id"` and change `RESOLUTION`, then rerun from the dataset cell. RF-DETR pose resolutions must be multiples of 24. To continue optimizer/epoch state instead, replace `weights=BASE_WEIGHTS` with `resume="/path/to/last.ckpt"` and keep the original resolution.